In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

data = [
    (1, 101, 250.0, "completed", "2025-01-10"),
    (2, 102, None, "pending", "2025-01-11"),
    (3, None, 300.0, "completed", "2025-01-12"),
    (4, 103, 450.0, None, "2025-01-13"),
    (5, 102, 200.0, "completed", None),
    (6, 104, None, "pending", "2025-01-15")
]

columns = ["order_id", "customer_id", "amount", "status", "order_date"]

df = spark.createDataFrame(data, columns)

1. Handling NULL values

In [0]:
df.filter(col("amount").isNull()).show()

df.filter(col("amount").isNotNull()).show()

2. fillna() — Replace NULL values

In [0]:
df.fillna({"amount": 0}).show()

In [0]:
df.fillna({
    "amount": 0,
    "status": "unknown",
    "customer_id": -1
}).show()

3. dropna() — Remove NULL records

In [0]:
df.dropna(subset=["customer_id"]).show()

In [0]:
df.show()
df.dropna().show()

Remove rows only when all selected columns are NULL:

The two main options:
Option	Behavior
how="any"	Remove row if any selected column is NULL
how="all"	Remove row only if all selected columns are NULL

In [0]:
df.dropna(
    subset=["customer_id", "amount"],
    how="any"
).show()

4. when() and otherwise() — Conditional transformations

In [0]:
from pyspark.sql.functions import when

df.withColumn(
    "order_category",
    when(col("amount") >= 400, "High")
    .when(col("amount") >= 200, "Medium")
    .otherwise("Low")
).show()

In [0]:
df.withColumn(
    "status",
    when(col("status").isNull(), "unknown")
    .otherwise(col("status"))
).show()

5. cast() — Change data types

In [0]:
from pyspark.sql.types import DoubleType

df.withColumn(
    "amount",
    col("amount").cast(DoubleType())
).show()

> - Useful functions:
> - Function	Purpose
> - trim()	Remove spaces from both sides
> - lower()	Convert to lowercase
> - upper()	Convert to uppercase
> - initcap()	Capitalize each word
> - length()	Return string length
> - split()	Split a string
> - concat()	Combine strings
> - regexp_replace()	Replace using regex
> - substring()	Extract part of a string

In [0]:
from pyspark.sql.functions import regexp_replace

df.withColumn(
    "phone",
    regexp_replace(col("phone"), "[^0-9]", "")
)

In [0]:
from pyspark.sql.functions import to_date, year, month, dayofmonth, datediff, current_date

df.withColumn(
    "order_date",
    to_date(col("order_date"), "yyyy-MM-dd")
)

df.withColumn(
    "order_month",
    month(col("order_date"))
).withColumn(
    "order_day_of_month",
    dayofmonth(col("order_date"))
).withColumn(
    "days_since_order",
    datediff(current_date(), col("order_date"))
).show()

### 8. Putting everything together: Silver Layer

In [0]:
from pyspark.sql import functions as F

silver_df = (
    df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    
    .dropDuplicates(["order_id"])
    
    .withColumn(
        "status",
        F.lower(F.trim(col("status")))
    )
    .withColumn(
        "status",
        F.when(F.col("status").isNull(), "unknown")
        .otherwise(col("status"))
    )
    .withColumn(
        "order_date",
        F.to_date(F.col("order_date"), "yyyy-MM-dd")
    )
)


display(silver_df)